# 24.1 设计推荐系统 / Design a Recommender System (YouTube / Netflix scale)

**中文**:欢迎来到最后一部分——**ML 系统设计**,大厂 senior DS / MLE 面试的重头戏。这类题(如"设计 YouTube 推荐")考的不是某个算法,而是你能否**把一个模糊的业务问题,拆解成一个端到端、能扛住亿级用户和百万级物品的机器学习系统**:澄清需求 → 定指标 → 设计架构 → 数据与特征 → 模型 → 服务与扩展 → 监控与陷阱。本部分每个系统设计题都遵循这套框架。推荐系统是第一课,也是最经典的一题。它的灵魂是一个几乎所有大规模推荐/搜索/广告系统都在用的骨架——**多阶段漏斗(检索 → 精排 → 重排)**。本节从零跑一个 demo,亲眼看到为什么不能"用一个模型给百万物品打分",以及漏斗如何把这变得可行。
**English**: Welcome to the final part — **ML system design**, the centerpiece of senior DS / MLE interviews at big tech. These questions (like "design YouTube recommendations") test not one algorithm but whether you can **decompose a vague business problem into an end-to-end machine-learning system that withstands billions of users and millions of items**: clarify requirements → define metrics → design architecture → data & features → models → serving & scaling → monitoring & pitfalls. Every system-design question in this part follows this framework. The recommender is the first lesson and the most classic question. Its soul is a skeleton used by nearly all large-scale recommendation/search/ads systems — the **multi-stage funnel (retrieval → ranking → re-ranking)**. This section runs a demo from scratch, showing why you can't "score millions of items with one model" and how the funnel makes it feasible.

---

**中文**:**系统设计面试的通用框架(每题都这样答)**:
**English**: **The universal system-design interview framework (answer every question this way)**:
1. **中文**:**澄清需求(clarify)**:问清楚——推荐给谁(用户)、推荐什么(视频/商品)、什么场景(首页/相关)、规模(用户数/物品数/QPS)、优化目标(观看时长?点击?留存?)、延迟要求。**别急着上模型,先问问题。**
   **Clarify**: ask — recommend to whom (users), what (videos/products), which surface (home/related), scale (users/items/QPS), objective (watch time? clicks? retention?), latency budget. **Don't jump to models; ask questions first.**
2. **中文**:**定指标(metrics)**:离线(Recall@k、NDCG)+ 在线(CTR、观看时长、留存)+ **护栏指标**(多样性、延迟、不推有害内容)。
   **Metrics**: offline (Recall@k, NDCG) + online (CTR, watch time, retention) + **guardrails** (diversity, latency, no harmful content).
3. **中文**:**架构(architecture)**:画出**多阶段漏斗**——这是核心。
   **Architecture**: draw the **multi-stage funnel** — this is the core.
4. **中文**:**数据与特征、模型、服务与扩展、监控与陷阱**:逐层展开。
   **Data & features, models, serving & scaling, monitoring & pitfalls**: expand each layer.

**中文**:**多阶段漏斗 —— 推荐系统的骨架**:
**English**: **The multi-stage funnel — the recommender's skeleton**:
- **中文**:**① 检索/召回(retrieval)**:从**百万级**物品里,用**极便宜**的方法(向量相似度 + ANN 近似最近邻索引)快速捞出**几百个**候选。要求:快、召回率高、能扛全量。模型:**双塔(two-tower)**(用户塔+物品塔,离线算好物品向量存 ANN 索引)、协同过滤、基于内容。
  **① Retrieval**: from **millions** of items, use a **very cheap** method (vector similarity + ANN approximate nearest neighbor index) to quickly fetch **hundreds** of candidates. Requirements: fast, high recall, handle the full catalog. Models: **two-tower** (user tower + item tower, item vectors precomputed into an ANN index), collaborative filtering, content-based.
- **中文**:**② 精排(ranking)**:对这几百个候选,用一个**昂贵但精准**的深度模型(丰富的用户/物品/上下文/交叉特征)打分排序。要求:准。模型:深度排序网络(Wide&Deep、DLRM、DCN)。
  **② Ranking**: score and rank those hundreds of candidates with an **expensive but precise** deep model (rich user/item/context/cross features). Requirement: accurate. Models: deep ranking nets (Wide&Deep, DLRM, DCN).
- **中文**:**③ 重排(re-ranking)**:对排好的 top 几十个,做**多样性(别全是一类)、新鲜度、商业规则、去重、打散**。要求:体验和业务。
  **③ Re-ranking**: on the ranked top-few-dozen, apply **diversity (not all one category), freshness, business rules, dedup, spreading**. Requirements: experience and business.

**中文**:**为什么必须分层**:精排模型每个物品要几毫秒,给百万物品打分要几分钟——用户等不了。检索用便宜方法把百万缩到几百,精排只打这几百个,漏斗让"大规模 + 精准 + 低延迟"同时成立。
**English**: **Why layering is mandatory**: a ranking model takes milliseconds per item, so scoring millions takes minutes — users won't wait. Retrieval cheaply narrows millions to hundreds, and ranking scores only those hundreds; the funnel makes "large-scale + accurate + low-latency" simultaneously possible.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 推荐系统设计, 逢面必考）**
> **中文**:**答题框架**:澄清需求(用户/物品/场景/规模/目标/延迟)→指标(离线 Recall@k/NDCG + 在线 CTR/观看时长/留存 + 护栏多样性/延迟)→**多阶段漏斗架构**→数据特征→模型→服务扩展→监控陷阱。**漏斗(核心)**:①**检索**(百万→几百, 便宜, 双塔+ANN 索引/CF/内容, 高召回)②**精排**(几百→几十, 昂贵深度模型+丰富交叉特征, 高精度, DLRM/DCN/Wide&Deep)③**重排**(多样性/新鲜度/商业规则/打散/去重)。**关键问题**:**冷启动**(新用户→热门/人口统计, 新物品→内容特征/探索)、**探索vs利用**(bandit/ε-greedy 别只推已知偏好)、**反馈回路/位置偏差**(用户只点看到的→用无偏 LTR/IPW 纠偏, 接 20.12/19)、**实时性**(近线特征更新)。**训练**:隐式反馈(点击/观看)、负采样、多目标(点击+时长+点赞, MMoE/ESMM 接 20.11)。**评估**:离线指标只是代理, 最终看**在线 A/B**(接 22.11)。**规模**:物品向量预计算存 ANN(Faiss/ScaNN), 特征存特征平台(22.9), 精排 GPU 服务(22.3/22.6)。面试金句:*"推荐系统用多阶段漏斗:双塔+ANN 检索把百万物品缩到几百(便宜高召回), 深度精排模型用丰富交叉特征打分(准), 重排做多样性和商业规则; 因为没法用重模型给百万物品打分, 漏斗让大规模+精准+低延迟同时成立; 关键难点是冷启动、探索利用、位置偏差/反馈回路, 多目标用 MMoE, 最终靠在线 A/B 评估。"*
> **English**: **Answer framework**: clarify requirements (users/items/surface/scale/objective/latency) → metrics (offline Recall@k/NDCG + online CTR/watch-time/retention + guardrail diversity/latency) → **multi-stage funnel architecture** → data & features → models → serving & scaling → monitoring & pitfalls. **The funnel (core)**: ① **retrieval** (millions → hundreds, cheap, two-tower + ANN index/CF/content, high recall) ② **ranking** (hundreds → dozens, expensive deep model + rich cross features, high precision, DLRM/DCN/Wide&Deep) ③ **re-ranking** (diversity/freshness/business rules/spreading/dedup). **Key issues**: **cold start** (new users → popular/demographic, new items → content features/exploration), **exploration vs exploitation** (bandit/ε-greedy, don't only serve known preferences), **feedback loops/position bias** (users click only what they see → unbiased LTR/IPW, per 20.12/19), **freshness** (near-line feature updates). **Training**: implicit feedback (clicks/watches), negative sampling, multi-objective (click + dwell + like, MMoE/ESMM per 20.11). **Evaluation**: offline metrics are just proxies; the final judge is **online A/B** (per 22.11). **Scale**: precompute item vectors into an ANN index (Faiss/ScaNN), store features in a feature store (22.9), serve ranking on GPU (22.3/22.6). Interview line: *"A recommender uses a multi-stage funnel: two-tower + ANN retrieval narrows millions of items to hundreds (cheap, high recall), a deep ranking model scores them with rich cross features (accurate), re-ranking handles diversity and business rules; since you can't score millions with a heavy model, the funnel makes large-scale + accurate + low-latency simultaneous; key challenges are cold start, exploration-exploitation, position bias/feedback loops, multi-objective via MMoE, and final evaluation by online A/B."*


In [ ]:

# ============================================================
# 核心机制:为什么必须用多阶段漏斗 / core: why the multi-stage funnel is mandatory
# 中文:百万物品。精排模型是一个真实的深度网络(每个物品要过网络)。演示"给全部物品打分"不可行,
#      而"检索缩小候选→只精排几百个"让延迟可接受。这是所有大规模推荐/搜索/广告的骨架。
# English: a million items. The ranking model is a real deep net (each item goes through the net). Show that "scoring
#      all items" is infeasible, while "retrieval narrows candidates → rank only hundreds" makes latency acceptable.
# ============================================================
import numpy as np, time
np.random.seed(0)
N_ITEMS, D = 1_000_000, 32
item_emb=np.random.randn(N_ITEMS, D).astype(np.float32)     # 物品向量(双塔的物品塔离线算好)/ item vectors (item tower, precomputed)
user_emb=np.random.randn(D).astype(np.float32)              # 用户向量(用户塔实时算)/ user vector (user tower, real-time)
# 精排 = 一个真实的深度排序网络(2 隐层, 用 用户||物品||交叉 特征)/ ranking = a real deep net (2 hidden layers, cross features)
W1=np.random.randn(3*D,128).astype(np.float32); W2=np.random.randn(128,64).astype(np.float32); W3=np.random.randn(64,1).astype(np.float32)
def deep_ranker(user, items):
    u=np.broadcast_to(user,(len(items),D))
    x=np.concatenate([u, items, u*items], 1)                # 交叉特征(真实排序器的关键)/ cross features
    h=np.maximum(x@W1,0); h=np.maximum(h@W2,0); return (h@W3).ravel()

# ✗ 朴素:用精排模型给全部 100 万物品打分 / naive: rank ALL 1M items with the deep model
t=time.time(); _=deep_ranker(user_emb, item_emb); naive_ms=(time.time()-t)*1000
# ✓ 漏斗:① 检索(便宜的向量相似度, 生产用 ANN 索引)取 top-500 → ② 精排只打这 500 个 / funnel
t=time.time()
sims=item_emb @ user_emb                                    # ① 检索:一次点积(生产是 ANN 亚毫秒)/ retrieval: dot product (ANN in prod)
cand=np.argpartition(-sims, 500)[:500]                      # 取 top-500 候选 / top-500 candidates
scores=deep_ranker(user_emb, item_emb[cand])               # ② 精排只打 500 个 / rank only 500
top10=cand[np.argsort(-scores)[:10]]                        # ③ 最终 top-10 / final top-10
funnel_ms=(time.time()-t)*1000
print(f"物品总数 / items: {N_ITEMS:,}")
print(f"✗ 精排全部物品 / rank ALL items:        {naive_ms:6.0f} ms   ← 请求时不可能(用户等不了)")
print(f"✓ 漏斗(检索 top-500 → 精排 500) / funnel: {funnel_ms:6.1f} ms   → 快 {naive_ms/funnel_ms:.0f}x, 延迟可接受")
print(f"最终 top-10 推荐 / final top-10 item ids:", top10[:10].tolist())
print("→ 检索(百万→几百, 便宜高召回)+ 精排(只打几百个, 昂贵高精度)= 多阶段漏斗, 大规模推荐的骨架")


In [ ]:

# ============================================================
# 可视化:推荐系统架构漏斗 / recommender architecture funnel
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5.5))
# ① 漏斗 / the funnel
ax[0].axis("off"); ax[0].set_title("推荐系统多阶段漏斗",fontsize=12,weight="bold")
stages=[("全部物品\n1,000,000","#BBBBBB",0.9),("① 检索 Retrieval\n(双塔+ANN, 便宜)\n→ ~500","#4C72B0",0.62),
        ("② 精排 Ranking\n(深度模型+交叉特征, 准)\n→ ~50","#55A868",0.34),("③ 重排 Re-rank\n(多样性/规则)\n→ top 10","#DD8452",0.08)]
widths=[0.8,0.55,0.32,0.16]
for i,((lab,c,y),w) in enumerate(zip(stages,widths)):
    ax[0].add_patch(plt.Rectangle((0.5-w/2,y),w,0.16,fc=c,alpha=0.5,ec=c,transform=ax[0].transAxes))
    ax[0].text(0.5,y+0.08,lab,ha="center",va="center",fontsize=8.5,transform=ax[0].transAxes)
    if i<3: ax[0].annotate("",xy=(0.5,y),xytext=(0.5,y-0.06+0.16),arrowprops=dict(arrowstyle="->"),transform=ax[0].transAxes)
ax[0].text(0.5,0.02,"每层:候选变少、模型变重、精度变高、延迟预算变松",ha="center",fontsize=8,style="italic",transform=ax[0].transAxes)
# ② 延迟对比 / latency comparison
ax[1].bar(["精排全部\n100万物品","多阶段\n漏斗"],[naive_ms,funnel_ms],color=["#C44E52","#55A868"])
for i,v in enumerate([naive_ms,funnel_ms]): ax[1].text(i,v+8,f"{v:.0f}ms",ha="center",fontsize=11,weight="bold")
ax[1].set_ylabel("请求延迟 ms"); ax[1].set_title(f"漏斗让延迟从不可行降到可接受({naive_ms/funnel_ms:.0f}x)")
plt.tight_layout(); plt.savefig("/tmp/sd01_viz.png",dpi=80); plt.show()
print("左:漏斗逐层收窄(百万→几百→几十→10), 模型逐层变重变准; 右:漏斗把延迟从数百ms降到十几ms")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **系统设计题的第一功力是"分解",而多阶段漏斗是推荐/搜索/广告的通用分解**:面试官问"设计 YouTube 推荐",新手会立刻跳到"用什么模型",而高手先画漏斗。为什么漏斗是标准答案?因为它同时解决了三个相互冲突的约束:**规模**(百万物品)、**精度**(要推得准)、**延迟**(几十毫秒内返回)。我们的 demo 一针见血:用精排深度模型给全部百万物品打分要几百毫秒(用户早跑了),而"便宜的检索把百万缩到几百 + 只对这几百个精排"把延迟压到十几毫秒。**这个"用便宜方法粗筛、用昂贵方法精选"的分层思想,不只推荐系统,而是几乎所有大规模 ML 系统(搜索、广告、内容审核)的通用骨架**——记住它,半数系统设计题的架构你就有了。
2. **每一层的目标和权衡都不同,这是深度的体现**:①**检索**追求**高召回**(别把好东西漏掉)+ 极致便宜(能扛全量),所以用双塔(离线把物品编码成向量存 ANN 索引,在线只需算用户向量 + 近似最近邻查询),而不是复杂模型;②**精排**追求**高精度**,候选已经少了,所以可以上最重的深度模型 + 最丰富的特征(用户历史、物品属性、上下文、交叉),不惜算力;③**重排**追求**体验和业务**——纯按精排分排序会让结果同质化(全是一类视频),所以要注入多样性、新鲜度、商业规则、去重打散。**能讲清"为什么每层用不同的模型和目标",比只会说"有个漏斗"深一个层次。**
3. **诚实的难点:推荐系统真正的魔鬼在数据和闭环,不在模型**。①**冷启动**:新用户没历史、新物品没交互——怎么推?靠热门兜底、人口统计特征、内容特征、以及**探索**(故意推一些不确定的,收集反馈)。②**反馈回路与位置偏差(最阴险)**:用户只能点击"你推给他看到的",所以你的训练数据被你自己的推荐污染了——模型倾向于强化已有偏好、把用户困在信息茧房,还会让"排在前面"被误当成"更相关"(接 20.12 位置偏差、19 因果)。不做无偏纠正(IPW/无偏 LTR)和探索(bandit),系统会越来越窄、越来越同质。③**多目标**:真实推荐不只优化点击,还要观看时长、点赞、留存、不推有害内容——这些目标常冲突(标题党提升点击但伤留存),要用多目标模型(MMoE/ESMM,接 20.11)和精心设计的价值函数平衡。④**离线≠在线**:离线 Recall/NDCG 涨了不代表线上指标涨(离线指标是代理),最终必须靠**在线 A/B**(接 22.11)判断。**结论:设计推荐系统的核心是多阶段漏斗(检索便宜高召回→精排昂贵高精度→重排多样性业务), 它让大规模+精准+低延迟同时成立; 但真正区分候选人水平的是能否讲清每层的不同目标、以及冷启动/位置偏差反馈回路/多目标/离线在线差异这些数据与闭环难题——模型只是系统的一部分。**

**English**:
1. **The first skill in system design is "decomposition," and the multi-stage funnel is the universal decomposition for recommendation/search/ads**: asked to "design YouTube recommendations," a novice jumps to "which model," an expert first draws the funnel. Why is the funnel the standard answer? Because it simultaneously resolves three conflicting constraints: **scale** (millions of items), **accuracy** (recommend well), **latency** (return within tens of milliseconds). Our demo cuts to it: scoring all million items with a deep ranker takes hundreds of milliseconds (the user is long gone), while "cheap retrieval narrows millions to hundreds + rank only those hundreds" compresses latency to ~15ms. **This "coarse-filter cheaply, fine-select expensively" layering is not just for recommenders but the universal skeleton of nearly all large-scale ML systems (search, ads, moderation)** — remember it and you have the architecture for half of system-design questions.
2. **Each layer's objective and tradeoff differ — showing this is depth**: ① **retrieval** pursues **high recall** (don't miss good items) + extreme cheapness (handle the full catalog), so it uses two-tower (offline encode items into vectors in an ANN index, online just compute the user vector + approximate nearest neighbor query), not a complex model; ② **ranking** pursues **high precision**; candidates are already few, so use the heaviest deep model + richest features (user history, item attributes, context, crosses), sparing no compute; ③ **re-ranking** pursues **experience and business** — pure ranking-score order makes results homogeneous (all one category of video), so inject diversity, freshness, business rules, dedup, spreading. **Being able to explain "why each layer uses different models and objectives" is a level deeper than just saying "there's a funnel."**
3. **Honest difficulty: a recommender's real devils are in data and the closed loop, not the model**. ① **Cold start**: new users have no history, new items no interactions — how to recommend? Via popularity fallback, demographic features, content features, and **exploration** (deliberately serve uncertain items to collect feedback). ② **Feedback loops and position bias (the most insidious)**: users can only click "what you recommended them to see," so your training data is polluted by your own recommendations — the model tends to reinforce existing preferences, trap users in filter bubbles, and mistake "ranked higher" for "more relevant" (per 20.12 position bias, 19 causal). Without unbiased correction (IPW/unbiased LTR) and exploration (bandits), the system narrows and homogenizes over time. ③ **Multi-objective**: real recommendation optimizes not just clicks but watch time, likes, retention, no harmful content — often conflicting (clickbait raises clicks but hurts retention), needing multi-objective models (MMoE/ESMM, per 20.11) and a carefully designed value function to balance. ④ **Offline ≠ online**: higher offline Recall/NDCG doesn't mean higher online metrics (offline metrics are proxies); the final judge must be **online A/B** (per 22.11). **Conclusion: designing a recommender centers on the multi-stage funnel (retrieval cheap high-recall → ranking expensive high-precision → re-ranking diversity/business), making large-scale + accurate + low-latency simultaneous; but what truly distinguishes candidates is explaining each layer's different objectives and the data/closed-loop challenges of cold start / position bias & feedback loops / multi-objective / offline-online gap — the model is only part of the system.**

> 💼 **实战视角 / Practical angle**
> **中文**:推荐系统落地:①**检索层**:双塔模型离线训练, 物品向量灌进 **ANN 索引(Faiss/ScaNN/HNSW)**, 在线算用户向量 + top-k 查询; 多路召回(协同+内容+热门+关注)取并集;②**精排层**:深度模型(DLRM/DCN/Wide&Deep), 特征来自**特征平台(22.9, point-in-time 防泄漏)**, GPU 服务(22.3/22.6);③**重排层**:多样性(MMR/DPP)、新鲜度、商业规则、去重;④**多目标**用 MMoE/ESMM(20.11);⑤**探索**用 bandit/ε-greedy 破反馈回路;⑥**评估**离线 Recall@k/NDCG 选模型, 上线 **A/B(22.11)** 看观看时长/留存/护栏;⑦**监控**漂移(22.10)、实时更新特征。**答题要点**:先澄清需求, 画漏斗, 逐层展开目标/模型/特征, 主动提冷启动/位置偏差/多目标/A/B。面试金句:*"设计推荐系统:多阶段漏斗——双塔+ANN 检索(百万→几百, 高召回便宜)、深度精排(交叉特征, 高精度)、重排(多样性/业务); 因为没法给百万物品跑重模型; 关键难点是冷启动(热门/内容/探索)、位置偏差和反馈回路(无偏 LTR+bandit)、多目标(MMoE)、离线只是代理最终看在线 A/B; 特征用特征平台防泄漏, 向量存 ANN 索引。"*
> **English**: Recommender in practice: ① **retrieval layer**: two-tower trained offline, item vectors loaded into an **ANN index (Faiss/ScaNN/HNSW)**, online compute user vector + top-k query; multi-source retrieval (collaborative + content + popular + follows) unioned; ② **ranking layer**: deep model (DLRM/DCN/Wide&Deep), features from a **feature store (22.9, point-in-time to prevent leakage)**, GPU serving (22.3/22.6); ③ **re-ranking layer**: diversity (MMR/DPP), freshness, business rules, dedup; ④ **multi-objective** via MMoE/ESMM (20.11); ⑤ **exploration** via bandit/ε-greedy to break feedback loops; ⑥ **evaluation** offline Recall@k/NDCG to pick models, online **A/B (22.11)** for watch time/retention/guardrails; ⑦ **monitor** drift (22.10), update features in near-real-time. **Answer keys**: clarify requirements first, draw the funnel, expand each layer's objective/model/features, proactively raise cold start/position bias/multi-objective/A/B. Interview line: *"Design a recommender: multi-stage funnel — two-tower + ANN retrieval (millions → hundreds, high recall cheaply), deep ranking (cross features, high precision), re-ranking (diversity/business); because you can't run a heavy model over millions; key challenges are cold start (popularity/content/exploration), position bias and feedback loops (unbiased LTR + bandit), multi-objective (MMoE), offline being just a proxy with online A/B the final judge; features from a feature store to prevent leakage, vectors in an ANN index."*

---
### 小结 / Summary
- **中文**:系统设计框架:澄清需求→指标(离线+在线+护栏)→架构→数据特征→模型→服务扩展→监控陷阱。
- **English**: System-design framework: clarify → metrics (offline + online + guardrails) → architecture → data & features → models → serving & scaling → monitoring & pitfalls.
- **中文**:推荐核心=多阶段漏斗:检索(双塔+ANN, 百万→几百, 便宜高召回)→精排(深度模型+交叉特征, 高精度)→重排(多样性/业务)。
- **English**: Recommender core = multi-stage funnel: retrieval (two-tower + ANN, millions → hundreds, cheap high-recall) → ranking (deep model + cross features, high precision) → re-ranking (diversity/business).
- **中文**:真正的难点在数据与闭环:冷启动、位置偏差/反馈回路、多目标(MMoE)、离线只是代理最终看在线 A/B。
- **English**: The real difficulty is data and the closed loop: cold start, position bias/feedback loops, multi-objective (MMoE), offline being just a proxy with online A/B as the final judge.
